In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

# Add root of repo to path to enable module imports
ROOT = Path(os.getcwd(), "..").resolve().as_posix()

if ROOT not in sys.path:
    sys.path.append(ROOT)

from config import Config
conf = Config()

DATA_DIR = Path(ROOT, conf.find("data.dir"))
DOTENV_PATH = Path(ROOT, "secrets/.env")

TARGET_COLUMNS = [
    "ACL",              # Anterior Cruciate Ligament tear / pathology
    "MCL",              # Medial Collateral Ligament tear / pathology
    "Medial Meniscus",  # Medial Meniscal tear / pathology
    "Lateral Meniscus", # Lateral Meniscal tear / pathology
    "Medial OA",        # Medial Compartment Osteoarthritis
    "Lateral OA",       # Lateral Compartment Osteoarthritis
    "PF OA",            # Patellofemoral Osteoarthritis
    "Effusion",         # Joint Effusion
    "Synovitis",        # Synovitis / Synovial Thickening
    "Baker's",          # Baker's Cyst
    "Contusion",        # Bone Marrow Edema / Contusion
    "Fracture",         # Bone Fracture
]

load_dotenv(DOTENV_PATH)

In [ ]:
train_df = pd.read_csv(DATA_DIR / conf.find("data.train"))

In [ ]:
train_df.info()

## Exploring duplicated Reports

TODO:
Need to see if reports are duplicated in studies for the same patient. Also need to consider the impact of duplicate reports on CLIP training

In [ ]:
dup_reports = pd.DataFrame(train_df[["StudyInstanceUID", "Report"]][train_df["Report"].duplicated(keep=False)])
dup_reports["Count"] = [1] * len(dup_reports)
agg_dup_reports = dup_reports.groupby(by="Report").agg({
    "Count": "sum",
    "StudyInstanceUID": lambda x: x
}).reset_index().rename(columns={"StudyInstanceUID": "StudyInstanceUIDs"})[["StudyInstanceUIDs", "Report", "Count"]]
agg_dup_reports

In [ ]:
translations_df = train_df.copy()
translations_df.insert(column="Original Language", value=pd.NA, loc=2)
translations_df.insert(column="Translated Report", value=pd.NA, loc=3)

## Do translations for 58 rows with Gold Labels

In [ ]:
from google.cloud import translate


def translate_text(
    text="¡Hola Mundo!",
    project_id="translation-v2-506402"
) -> tuple[str, str] | str:
    """
    Returns:
        Tuple of (translated text, detected_language)
    """

    client = translate.TranslationServiceClient()
    location = "global"
    parent = f"projects/{project_id}/locations/{location}"

    try:
        response = client.translate_text(
            request={
                "parent": parent,
                "contents": [text],
                "mime_type": "text/plain",
                "target_language_code": "en",
            }
        )
        # for translation in response.translations:
        #     print("Translated text: {}".format(translation.translated_text))

        translation = response.translations[0]
        return translation.translated_text, translation.detected_language_code

    except Exception as e:
        print(f"[ERROR] Error during translation: {str(e)}")
        return e

In [ ]:
translate_text()

In [ ]:
gold_df = pd.read_csv(DATA_DIR / conf.find("data.gold_label"))
print(f"Row count: {len(gold_df)}")


if len(gold_df[gold_df[TARGET_COLUMNS].isna().any(axis=1)]) == 0:
    pass
else:
    print("[WARNING] There are rows missing labels!")

In [ ]:
gold_translated = gold_df.copy()
gold_translated.insert(column="Original Language", value=pd.NA, loc=2)
gold_translated.insert(column="Translated Report", value=pd.NA, loc=3)

In [ ]:
for i in range(len(gold_translated)):
    print(f"Translating row {i + 1:>3} of {len(gold_translated)}")
    report = gold_translated.loc[i, "Report"]
    translated_report, detected_lang_code = translate_text(report)
    gold_translated.loc[i, "Translated Report"] = translated_report
    gold_translated.loc[i, "Original Language"] = detected_lang_code
    gold_translated.to_csv(DATA_DIR / conf.find("data.gold_translated"))


In [ ]:
gold_translated